# Setup

In [ ]:
suggestive_t=1e-5
# gw_t = 0.05/489673 # for 0.3
gw_t = 0.05/334605 # for 0.2
g_alpha = 0.05/495465
apoe_str_id = 'chr19:44909968:GGGGGGGGGGG:GGGGGGGGG'
apoe_snp_id = '19_45411941_rs429358_T_C'

In [ ]:
# code from other analyses I will need probably
## imports
import sys
import os

from scipy import stats

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

font_kwargs={
    'fontsize':18
}
sns.set_theme(font_scale=1.2, context='paper', style='white', palette='tab10')
# set default stuff
plink_keys=['#CHROM', "POS"]
meta_keys=['CHR', "BP"]

scatter_palette={
    'SNP':'tab:blue', 
    'STR':'tab:orange', 
    'adjusted':'tomato', 
    'unadjusted':'teal'
}
custom_palette = sns.color_palette("tab10", 4)
# manhatti_palette = sns.color_palette()[:4]
manhatti_palette = {}
for c in range(1, 23):
    manhatti_palette[c] = custom_palette[c % len(custom_palette)]

# methods
def get_results(file_path, is_meta=False, cut=True, cut_threshold=10):
    df = pd.read_csv(file_path, sep="\t")
    initial = df.shape[0]
    df = df.dropna(axis=0)
    after = df.shape[0]
    print(f'drop {initial-after} rows because of NA values')
    df = df.astype({'P':float})
    df.loc[df.P<sys.float_info.min, 'P']=sys.float_info.min

    min_ct = df[df.P == sys.float_info.min].shape[0]
    if min_ct > 0:
        print(f'set {min_ct} P values to maximum possible min of {sys.float_info.min}')
    df['-log10'] = -np.log10(df.P)

    if not is_meta:
        df = df[df['TEST']=='ADD']

    if cut:
        df=df[df['-log10'] < cut_threshold]
    
    return df

def manhatti(df, title, sort_keys=['#CHROM', "POS"], hue_key='#CHROM', style_key=None, y_steps=10, legend=None, 
             sort_key='-log10', sort_asc=True, plot_y_thresh=True, do_full=True, suggest_t=5, p_sig_t=-np.log10(gw_t)):
    plot_df = df.copy(deep=True)
    plt.figure(figsize=(30,10))
    if do_full:
        chr_maxxs = plot_df.groupby('#CHROM')['POS'].max()
        chr_starts = {plot_df['#CHROM'].min() : 0}
        for c in range(2, 23):
            chr_starts[c] = chr_starts[c-1] + chr_maxxs[c-1] + 1
        chr_starts
        plot_df['i'] = [p + chr_starts[c] for p,c in zip(plot_df['POS'], plot_df['#CHROM'])]
    else:
        plot_df = df.sort_values(sort_keys)
        plot_df = plot_df.reset_index(drop=True); 
        plot_df['i'] = plot_df.index

    plot_df = plot_df.sort_values(by=sort_key, ascending=sort_asc)
    plot = sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhatti_palette, legend=legend, linewidth=0.3, style=style_key)
    labels_df=plot_df.groupby(sort_keys[0])['i'].median()
    plot.set_xlabel('Chromosome', fontdict=font_kwargs)
    plot.set_ylabel('-log10(p)', fontdict=font_kwargs)
    plot.set_xticks(labels_df)
    plot.set_xticklabels(labels_df.index)
    max_x = plot_df.i.max()
    max_x = max_x + 0.01*max_x
    min_x = plot_df.i.min() - 0.01*max_x
    plt.xlim([min_x, max_x])

    max_y=int(max(plot_df['-log10'].max(), y_steps))+1
    step = int(max_y/y_steps)
    
    plot.set_yticks(range(1, max_y, step))
    if plot_y_thresh:
        plt.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, colors=['tab:red', 'k'], linestyles='dashed')

    #plot.grid(axis='y')
    #plot.set_title(title)
    plt.grid(axis='y')
    plt.title(title, fontdict=font_kwargs)
    plt.tight_layout()

    return plot.figure

In [ ]:
def broken_manhatti(df, title, sort_keys=['#CHROM', "POS"], hue_key='#CHROM', style_key=None, legend=None, 
             sort_key='-log10', sort_asc=True, plot_y_thresh=True, do_full=True, suggest_t=5, p_sig_t=-np.log10(gw_t), plot_break=(30,100)):
    plot_df = df.copy(deep=True)
    f, (outlier_ax, plot_ax) = plt.subplots(2, 1, sharex=True, figsize=(30,10), height_ratios=[1,8])
    if do_full:
        chr_maxxs = plot_df.groupby('#CHROM')['POS'].max()
        chr_starts = {plot_df['#CHROM'].min() : 0}
        for c in range(2, 23):
            chr_starts[c] = chr_starts[c-1] + chr_maxxs[c-1] + 1
        chr_starts
        plot_df['i'] = [p + chr_starts[c] for p,c in zip(plot_df['POS'], plot_df['#CHROM'])]
    else:
        plot_df = df.sort_values(sort_keys)
        plot_df = plot_df.reset_index(drop=True); 
        plot_df['i'] = plot_df.index

    plot_df = plot_df.sort_values(by=sort_key, ascending=sort_asc)
    plot = sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhatti_palette, legend=legend, linewidth=0.3, style=style_key, ax=plot_ax)
    
    labels_df=plot_df.groupby(sort_keys[0])['i'].median()
    plot.set_xlabel('Chromosome', fontdict=font_kwargs)
    plot.set_ylabel('-log10(p)', fontdict=font_kwargs)

    plot_ax.set_xticks(labels_df)
    plot_ax.set_xticklabels(labels_df.index)
    max_x = plot_df.i.max()
    max_x = max_x + 0.01*max_x
    min_x = plot_df.i.min() - 0.01*max_x
    plt.xlim([min_x, max_x])
    plot_ax.set_yticks(range(0, 30, 5))

    if plot_y_thresh:
        plt.hlines(y=[p_sig_t, suggest_t], xmin=min_x, xmax=max_x, colors=['tab:red', 'k'], linestyles='dashed')
    plt.grid(axis='y')
        # plt.tight_layout()

    sns.scatterplot(plot_df, x='i', y='-log10', hue=hue_key, palette=manhatti_palette, legend=legend, linewidth=0.3, style=style_key, ax=outlier_ax)
    y_max_outliers = outlier_ax.get_ylim()[1]
    y_shift = y_max_outliers + y_max_outliers*0.1
    outlier_ax.set_ylim(bottom=plot_break[1], top=y_shift)  # outliers only
    plot_ax.set_ylim(0, plot_break[0])  # most of the data

    outlier_ax.spines["bottom"].set_visible(False)
    plot_ax.spines["top"].set_visible(False)

    # outlier_ax.xaxis.tick_top()
    outlier_ax.tick_params(top=False, bottom=False)

    d = .25  # proportion of vertical to horizontal extent of the slanted line
    ax_kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
                linestyle="none", color='k', mec='k', mew=1, clip_on=False)
    outlier_ax.plot([0, 1], [0, 0], transform=outlier_ax.transAxes, **ax_kwargs)
    outlier_ax.set_ylabel('')
    plot_ax.plot([0, 1], [1, 1], transform=plot_ax.transAxes, **ax_kwargs)



    plt.subplots_adjust(hspace=0.05)
    outlier_ax.set_title(title, **font_kwargs)
    plt.tight_layout()

    return plot.figure

In [ ]:
def check_mkdir(dir_path):

    if not os.path.isdir(dir_path):
        os.makedirs(dir_path)

In [ ]:
def plot_corrs(df, x='OR_t0', y='OR_t1', title='', verbose=True):
    sns.scatterplot(df, x=x, y=y)
    # df_cp = df.dropna(axis=0).copy(deep=True)
    df_cp = df.replace(np.nan, 0).copy(deep=True)
    stat_out = stats.pearsonr(df_cp[x],df_cp[y])
    # plt.text(0.05, 0.95, 'Pearsons\'s', transform=plt.gca().transAxes)
    figtext = 'Pearson\'s rho={:.4f}'.format(stat_out.statistic) #+ ';  p={:.2e}'.format(stat_out.pvalue)
    plt.text(0.05, 0.95, figtext, transform=plt.gca().transAxes)
    plt.title(title)
    if verbose:
        print(stat_out)

In [ ]:
results_path='/data/str_gwa/data/eval/age_adjusted'
check_mkdir(results_path)
APOE_POS=44908684

In [ ]:
initial_str_df = pd.read_csv(
    f'{results_path}/../initial_str_results_biallelic.csv', sep='\t'
    ).sort_values(by=['#CHROM', 'POS'])
print(f'Initial study STR results: n={len(initial_str_df)}')
initial_str_df.head()

In [ ]:
plotrun = False

# Load age adjusted

## Overview

In [ ]:
age_adjust_df = get_results(
    f'{results_path}/age_adjusted_str_biallelic.glm.linear' , cut=False
).sort_values(by=['#CHROM', 'POS'])
print(f'Age adjusted STR results: n={len(age_adjust_df)}')
age_adjust_df.head()

In [ ]:
if plotrun:
    mask = (age_adjust_df.P < 0.05) & (age_adjust_df.P > 1e-30)
    p = manhatti(
        age_adjust_df[mask], "Age adjusted STR GWAS results"
    )
    p.savefig(f'{results_path}/age_adjusted_str_manhatti.png', dpi=300)
    

In [ ]:
if plotrun:
    mask = (initial_str_df.P < 0.05) & (initial_str_df.P > 1e-30)
    p = manhatti(
        initial_str_df[mask], "Initial STR GWAS results"
    )
    p.savefig(f'{results_path}/initial_str_manhatti_for_reference.png', dpi=300)

In [ ]:
if plotrun: 
    p = broken_manhatti(age_adjust_df, title='Age adjusted STR GWAS results')
    p.savefig(f'{results_path}/age_adjusted_str_manhattan.png', dpi=300)

In [ ]:
initial_key = 'without age adjustment'
age_key = 'with age adjustment'
initial_str_df['source'] = initial_key
age_adjust_df['source'] = age_key
age_combined = pd.concat([initial_str_df, age_adjust_df], ignore_index=True)

In [ ]:
if plotrun:
    manhatti_palette[age_key] = custom_palette[1]
    manhatti_palette[initial_key] = custom_palette[0]
    mask = (age_combined.P < 0.05) & (age_combined.P > 1e-30)
    p = manhatti(age_combined[mask], 'Comparison of STRs with and without age adjustment', hue_key='source', style_key='#CHROM', sort_key='source', legend='auto')
    p.savefig(f'{results_path}/comparison_age_adjusted_str_manhatti.png', dpi=300)

## Correlate

In [ ]:
merged = initial_str_df.merge(age_adjust_df, on=['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'A1'], suffixes=('_initial', '_age_adjust'), how='inner')
print(merged.shape)
merged.head()

In [ ]:
title = 'Correlation of BETA-Values without adjustment for age. \n All variants'
plot_corrs(merged, x='BETA_initial', y='BETA_age_adjust', title=title)
plt.savefig(f'{results_path}/correlation_beta_values_all_variants.png', dpi=300, bbox_inches='tight')

In [ ]:
mask = (merged.P_initial < 0.05)
title = 'Correlation of beta-Values with vs. without adjustment for age.\n At least nominally significant variants in the initial study'
plot_corrs(merged[mask], x='BETA_initial', y='BETA_age_adjust', title=title)
if plotrun:
    plt.savefig(f'{results_path}/correlation_beta_values_nominally_variants.png', dpi=300, bbox_inches='tight')

In [ ]:
mask = (merged.P_initial < suggestive_t)
title = 'Correlation of beta-Values with vs. without adjustment for age.\n At least suggestively significant variants in the initial study'
plot_corrs(merged[mask], x='BETA_initial', y='BETA_age_adjust', title=title)
if plotrun:
    plt.savefig(f'{results_path}/correlation_beta_values_suggestive_variants.png', dpi=300, bbox_inches='tight')

In [ ]:
mask = (merged.P_initial < suggestive_t)
title = 'Correlation of P-Values with vs. without adjustment for age.\n At least suggestively significant variants in the initial study'
plot_corrs(merged[mask], x='P_initial', y='P_age_adjust', title=title)
if plotrun:
    plt.savefig(f'{results_path}/correlation_p_values_suggestive_variants.png', dpi=300, bbox_inches='tight')

## Count Vars

In [ ]:
sig_initial = initial_str_df[initial_str_df.P < suggestive_t].ID
sig_age_adjust = age_adjust_df[age_adjust_df.P < suggestive_t].ID
overlap_ids = set(sig_initial).intersection(set(sig_age_adjust))
print(f'Number of suggestively significant STRs in initial study: {len(sig_initial)}')
print(f'Number of suggestively significant STRs in age adjusted study: {len(sig_age_adjust)}')
print(f'Number of overlapping suggestively significant STRs: {len(overlap_ids)}') 

In [ ]:
# Check lost ones in initial study
mask = sig_initial.isin(overlap_ids) == False
lost_ids = sig_initial[mask]
merged[merged.ID.isin(lost_ids)][
    ['ID', 'BETA_initial', 'P_initial', '-log10_initial', 'BETA_age_adjust', 'P_age_adjust', '-log10_age_adjust']
].sort_values(by='-log10_initial')

In [ ]:
# Check lost ones in age adjusted study
mask = sig_age_adjust.isin(overlap_ids) == False
lost_ids = sig_age_adjust[mask]
merged[merged.ID.isin(lost_ids)][
    ['ID', 'BETA_initial', 'P_initial', '-log10_initial', 'BETA_age_adjust', 'P_age_adjust', '-log10_age_adjust']
].sort_values(by='-log10_initial')

In [ ]:
# Count genome-wide significant ones
(merged.P_age_adjust < gw_t).sum(), (merged.P_initial < gw_t).sum()

In [ ]:
# Assess lost ones in genome-wide significants
sig_initial = initial_str_df[initial_str_df.P < gw_t].ID
sig_age_adjust = age_adjust_df[age_adjust_df.P < gw_t].ID
overlap_ids = set(sig_initial).intersection(set(sig_age_adjust))
print(f'Number of genome-wide significant STRs in initial study: {len(sig_initial)}')
print(f'Number of genome-wide significant STRs in age adjusted study: {len(sig_age_adjust)}')
print(f'Number of overlapping genome-wide significant STRs: {len(overlap_ids)}') 

## Calculate Information Gain

In [ ]:
def get_maxes_idxs(df, winsize=250000):
    keep=[]

    for i, r in df.iterrows():
        chr = r['#CHROM']
        pos = r.POS
        cands = df[(df['#CHROM'] == chr) 
                & (abs(df.POS-pos)<winsize)]
        min_idx = cands.P.idxmin()
        if min_idx == i: 
            keep.append(i)

    return keep

In [ ]:
max_idxs = get_maxes_idxs(age_adjust_df[age_adjust_df.P<suggestive_t], winsize=250000)
age_adjust_df['islocmax'] = np.where(age_adjust_df.index.isin(max_idxs), 1, 0)
age_adjust_df.islocmax.value_counts()

In [ ]:
mask = initial_str_df.islocmax == 1
lead_ids = set(initial_str_df[mask].ID)
lead_ids.update(set(age_adjust_df[age_adjust_df.islocmax==1].ID))
len(lead_ids)

In [ ]:
merged_leads = merged[merged.ID.isin(lead_ids)].copy()
merged_leads['se_ratio'] = merged_leads.SE_age_adjust / merged_leads.SE_initial
merged_leads.se_ratio.describe()

In [ ]:
gain = (1-merged_leads.se_ratio)*100
gain.describe()

### for all

In [ ]:
merged['se_ratio'] = merged.SE_age_adjust / merged.SE_initial
merged.se_ratio.describe().apply(lambda x: f"{x:.7f}")

In [ ]:
merged.iloc[[merged.se_ratio.argmax(), merged.se_ratio.argmin()]][
    ['ID', 'P_initial', 'P_age_adjust', 'se_ratio', '-log10_initial', '-log10_age_adjust', 'SE_initial', 'SE_age_adjust']
]

In [ ]:
gain = (1-merged.se_ratio)*100
gain.describe().apply(lambda x: f"{x:.7f}")